In [0]:
# Silver Layer: Order Items — Cleansing + CDC MERGE

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, upper, to_timestamp, when,
    regexp_replace, coalesce, lit, current_timestamp,
    row_number, desc
)
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE = "ecomstore.ecomlake.bronze_order_items"
SILVER_TABLE = "ecomstore.ecomlake.silver_order_items"
TEMP_STAGING_TABLE = "ecomstore.ecomlake.temp_order_items"

# Read from Bronze
bronze_df = spark.read.table(BRONZE_TABLE)

# Deduplication
w = Window.partitionBy("order_item_id").orderBy(desc("_ingestion_timestamp"))

deduped_df = (
    bronze_df
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Cleansing & Standardization 
def clean_amount(c):
    return (
        when(col(c).isNotNull(),
             regexp_replace(col(c).cast("string"), r"[^\d.]", "").cast(DoubleType()))
        .otherwise(None)
    )

cleaned_df = (
    deduped_df
    .withColumn("order_item_id", trim(upper(col("order_item_id"))))
    .withColumn("order_id",      trim(upper(col("order_id"))))
    .withColumn("product_id",    trim(upper(col("product_id"))))
    .withColumn("seller_id",     trim(upper(col("seller_id"))))
    .withColumn("quantity",
        when(col("quantity").cast(DoubleType()).cast(IntegerType()) > 0,
             col("quantity").cast(DoubleType()).cast(IntegerType()))
        .otherwise(None)
    )
    .withColumn("unit_price",      clean_amount("unit_price"))
    .withColumn("total_price",     clean_amount("total_price"))
    .withColumn("discount_pct", clean_amount("discount_pct"))
    .withColumn("unit_price",
        when(col("unit_price") < 0, None).otherwise(col("unit_price")))
    .withColumn("total_price",
        when(col("total_price") < 0, None).otherwise(col("total_price")))
    .withColumn("status",
        when(col("status").isin("active", "cancelled", "returned"), col("status"))
        .when(col("status").isNull(), lit("active"))
        .otherwise(lit("unknown"))
    ) 
    .withColumn("created_at",
        coalesce(to_timestamp(col("created_at")), to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss")))
    .withColumn("_silver_processed_at", current_timestamp())
    .filter(col("order_item_id").isNotNull() & col("order_id").isNotNull())
)

# Select final Silver schema
silver_df = cleaned_df.select(
    "order_item_id", "order_id", "product_id", "seller_id",
    "quantity", "unit_price", "discount_pct", "total_price",
    "status", "created_at",
    "_batch_id", "_silver_processed_at"
)

# Disk Staging to prevent Double Compute
silver_df.write.format("delta").mode("overwrite").saveAsTable(TEMP_STAGING_TABLE)
staged_silver_df = spark.read.table(TEMP_STAGING_TABLE)

# Data Quality split
# Use the STAGED data, not the raw transformation logic
good_records = staged_silver_df.filter(
    col("order_item_id").isNotNull() &
    col("order_id").isNotNull() &
    col("total_price").isNotNull()
)

bad_records = staged_silver_df.filter(
    col("order_item_id").isNull() |
    col("order_id").isNull() |
    col("total_price").isNull()
)

# Log bad records to quarantine
(
    bad_records
    .withColumn("rejection_reason",
        when(col("order_item_id").isNull(), lit("null_order_item_id"))
        .when(col("order_id").isNull(),     lit("null_order_id"))
        .otherwise(lit("null_total_price"))
    )
    .write.format("delta").mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("ecomstore.ecomlake.silver_quarantine_order_items")
)

# CDC MERGE
if spark.catalog.tableExists(SILVER_TABLE):
    silver_target = DeltaTable.forName(spark, SILVER_TABLE)

    (
        silver_target.alias("target")
        .merge(
            good_records.alias("source"),
            "target.order_item_id = source.order_item_id"
        )
        .whenMatchedUpdate(
            condition="source.status != target.status OR source.total_price != target.total_price",
            set={
                "status":               "source.status",
                "quantity":             "source.quantity",
                "unit_price":           "source.unit_price",
                "discount_pct":         "source.discount_pct",
                "total_price":          "source.total_price",
                "_batch_id":            "source._batch_id",
                "_silver_processed_at": "source._silver_processed_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"✅ CDC MERGE complete on {SILVER_TABLE}")

else:
    (
        good_records
        .write
        .format("delta")
        .mode("overwrite")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"✅ Silver order_items table created: {SILVER_TABLE}")

# Clean up staging table
spark.sql(f"DROP TABLE IF EXISTS {TEMP_STAGING_TABLE}")

